In [ ]:
import yaml
from pathlib import Path

current_dir = Path.cwd()

# Construct your relative path: current.parent.parent/config/sequences.yaml
config_path = current_dir.parent / "config" / "sequences.yaml"

print(f"Looking for config at: {config_path.absolute()}")
print(f"File exists: {config_path.exists()}")

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_yaml(file_path: Path) -> dict:
    with open(file_path, "r") as f:
        return yaml.safe_load(f)
    
seq_source_path = config_path.parent / "sequences.yaml"
all_sequences = load_yaml(seq_source_path) 
seqs = [data["sequence"] for name, data in all_sequences.items()]
print(seqs)

Looking for config at: /home/andres/Research/idp-ml/config/sequences.yaml
File exists: True
['GPGMRGKVKWFDSKKGYGFITKDEGGDVFVHWSAIEMEGFKTLKEGQVVEFEIQEGKKGGQAAHVKV', 'GSHCFLDGIDKAQEEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLKGEAMHGQVDC', 'GPSDAAVDTSSEITTKDLKEKKEVVEEAENGRDAPANGNAENEENGEQEADNEVDEECEEGGEEEEEEEEGDGEEEDGDEDEEAESATGKRAAEDDEDDDVDTKKQKTDEDD', 'MAHHHHHHSAALEVLFQGPMSDAAVDTSSEITTKDLKEKKEVVEEAENGRDAPANGNANEENGEQEADNEVDEECEEGGEEEEEEEEGDGEEEDGDEDEEAESATGKRAAEDDEDDDVDTKKQKTDEDD', 'KLKEANKQQNFNTGIKDFDFWLSEVEALLASEDYGKDLASVNNLLKKHQLLEADISAHEDRLKDLNSQADSLMTSSAFDTSQVKDKRETINGRFQRIKSMAAARRAKLNESHRL', 'RLEESLEYQQFVANVEEEEAWINEKMTLVASEDYGDTLAAIQGLLKKHEAFETDFTVHKDRVNDVAANGEDLIKKNNHHVENITAKMKGLKGKVSDLEKA', 'SSFHRIIPGFMSQGGDFTRHNGTGGKSIYGEKFEDENFILKHTGPGILSMANAGPNTNGSQFFISTAKTEFLDGKHVVFGKVKEGMNIVEAMERFGSRNGKTSKKITIADSGQLE', 'MEEVTIKANLIFANGSTQTAEFKGTFEKATSEAYAYADTLKKDNGEWTVDVADKGYTLNIKFAG', 'GTQNRPLLRNSLDDLVGPPSNLEGQSDERALLDQLHTLLSNTDATGLEEIDRALGIPELVNQGQALEPKQD', 'MVPAHKLDSPTMSRARIGSDPLAYEPKEDLPVITIDPA

In [36]:
import jax
import jax.numpy as jnp
from jax.scipy.linalg import toeplitz
from functools import partial

MASS = 0
CHARGE = 1
SIGMA = 2
HPS1 = 3
HPS2 = 4

AA_PARAMS = {
    "-": dict(mass=0.0, charge=0.0, sigma=0.0, HPS1=0.0, HPS2=0.0),  # Padding
    "A": dict(mass=71.08, charge=0.0, sigma=0.504, HPS1=0.730, HPS2=0.003),
    "R": dict(mass=156.20, charge=1.0, sigma=0.656, HPS1=0.000, HPS2=0.723),
    "N": dict(mass=114.10, charge=0.0, sigma=0.568, HPS1=0.432, HPS2=0.160),
    "D": dict(mass=115.10, charge=-1.0, sigma=0.558, HPS1=0.378, HPS2=0.002),
    "C": dict(mass=103.10, charge=0.0, sigma=0.548, HPS1=0.595, HPS2=0.400),
    "Q": dict(mass=128.10, charge=0.0, sigma=0.602, HPS1=0.514, HPS2=0.468),
    "E": dict(mass=129.10, charge=-1.0, sigma=0.592, HPS1=0.459, HPS2=0.022),
    "G": dict(mass=57.05, charge=0.0, sigma=0.450, HPS1=0.649, HPS2=0.784),
    "H": dict(mass=137.10, charge=0.5, sigma=0.608, HPS1=0.514, HPS2=0.487),
    "I": dict(mass=113.20, charge=0.0, sigma=0.618, HPS1=0.973, HPS2=0.687),
    "L": dict(mass=113.20, charge=0.0, sigma=0.618, HPS1=0.973, HPS2=0.335),
    "K": dict(mass=128.20, charge=1.0, sigma=0.636, HPS1=0.514, HPS2=0.095),
    "M": dict(mass=131.20, charge=0.0, sigma=0.618, HPS1=0.838, HPS2=0.993),
    "F": dict(mass=147.20, charge=0.0, sigma=0.636, HPS1=1.000, HPS2=0.871),
    "P": dict(mass=97.12, charge=0.0, sigma=0.556, HPS1=1.000, HPS2=0.471),
    "S": dict(mass=87.08, charge=0.0, sigma=0.518, HPS1=0.595, HPS2=0.487),
    "T": dict(mass=101.10, charge=0.0, sigma=0.562, HPS1=0.676, HPS2=0.274),
    "W": dict(mass=186.20, charge=0.0, sigma=0.678, HPS1=0.946, HPS2=0.753),
    "Y": dict(mass=163.20, charge=0.0, sigma=0.646, HPS1=0.865, HPS2=0.984),
    "V": dict(mass=99.07, charge=0.0, sigma=0.586, HPS1=0.892, HPS2=0.428),
}

# ── Derived arrays ───────────────────────────────────
# Protein + number id dictionary
ids = {aa: i for i, aa in enumerate(list(AA_PARAMS.keys()))}

param_matrix = jnp.array([
    [v["mass"], v["charge"], v["sigma"], v["HPS1"], v["HPS2"]] 
    for v in AA_PARAMS.values()
])

is_charged_mask = jnp.array([v["charge"] != 0 for v in AA_PARAMS.values()])

def s_all(window_ids):
    counts = jnp.bincount(window_ids, length=21)
    real_counts = counts[1:] 
    
    N_all = jnp.sum(real_counts)
    
    p_all = real_counts / jnp.where(N_all == 0, 1.0, N_all)
    safe_p_all = jnp.where(p_all > 0, p_all, 1.0)
    
    S_all = -jnp.sum(p_all * jnp.log2(safe_p_all))

    return jnp.where(N_all == 0, 0.0, S_all)

def s_q(window_ids):
    counts = jnp.bincount(window_ids, length=21)
    charged_counts = counts * is_charged_mask
    
    N_q = jnp.sum(charged_counts)
    
    p_q = charged_counts / jnp.where(N_q == 0, 1.0, N_q)
    safe_p_q = jnp.where(p_q > 0, p_q, 1.0)
    
    S_q = -jnp.sum(p_q * jnp.log2(safe_p_q))

    return jnp.where(N_q == 0, 0.0, S_q)

#sequence charge decoration (SCD)
def scd(window): 
    k = len(window)
    qij = jnp.outer(window,window)
    dist = toeplitz(jnp.arange(k)) ** 0.5
    S = qij * dist
    s_masked = jnp.tril(S,k=-1)
    return jnp.sum(s_masked)/k
#sequence hydropathy decoration (ShD)
def shd(window): 
    k = len(window)
    lambda_ij = window[:,None] + window[None,:]
    dist = toeplitz(jnp.arange(k))
    safe_dist = jnp.where(dist == 0, 1.0, dist)
    S = lambda_ij * (1/safe_dist)
    s_masked = jnp.tril(S,k=-1)
    return jnp.sum(s_masked)/k

def net_charge(window):
    return jnp.sum(window)

#Fraction of Charged Residues (FCR)
def fcr(window):
    return jnp.sum(jnp.abs(window) > 0) / len(window)

def charge_asymmetry(window):
    n = len(window)
    # Calculate fractions of positive and negative charges
    pos = jnp.sum(window > 0) / n
    neg = jnp.sum(window < 0) / n
    total = pos + neg
    
    # jnp.where(condition, value_if_true, value_if_false)
    return jnp.where(total == 0, 0.0, (pos - neg)**2 / total)

#Mean Hydrophobicity (using HPS1)
def mean_hydro(window):
    return jnp.mean(window)

def sbcs(window_lambda):
    N = len(window_lambda)
    
    # 1. Boolean mask of high-lambda residues
    is_high = window_lambda > 0.5
    
    # 2. The "Consecutive" Trick using Cumulative Sum
    # If we take the cumsum, the difference between the cumsum at index j 
    # and index i will be EXACTLY 1 if they are consecutive high-lambda residues.
    C = jnp.cumsum(is_high)
    C_diff = C[None, :] - C[:, None]
    idx = jnp.arange(N)
    dist_matrix = idx[None, :] - idx[:, None]
    
    # 4. Define the valid pairs (j > i)
    # - Both i and j must be high lambda: jnp.outer(is_high, is_high)
    # - They must be consecutive: C_diff == 1
    # - We only look at the upper triangle: dist_matrix > 0
    valid_pairs = (C_diff == 1) & jnp.outer(is_high, is_high) & (dist_matrix > 0)
    
    # 5. Calculate inverse distances safely
    # Prevent division by zero on the diagonal (even though valid_pairs masks it later, 
    # JAX evaluates all math, so 1/0 creates a NaN that can pollute gradients)
    safe_dist = jnp.where(dist_matrix == 0, 1.0, dist_matrix)
    inv_dist = jnp.where(valid_pairs, 1.0 / safe_dist, 0.0)
    
    # 6. Sum them up and multiply by the mean lambda of the window
    sum_inv_dist = jnp.sum(inv_dist)
    mean_lambda = jnp.mean(window_lambda)
    
    return mean_lambda * sum_inv_dist

def conv(x, f, kernel_size):
    # This creates the (Num_Windows, Window_Size) matrix
    indices = jnp.arange(len(x)- kernel_size + 1)[:, None] + jnp.arange(kernel_size)
    windows = x[indices]
    return jax.vmap(f)(windows)

batch_conv = jax.vmap(conv, in_axes=(0, None, None))

# convolution example

def create_bucket_batch(protein_list, bucket_size, param_matrix, ids):
    """
    Takes a list of strings, pads them to bucket_size, 
    and returns features + mask.
    """
    batch_features = []
    batch_masks = []
    batch_ids = []
    
    for seq in protein_list:
        # 1. Convert to IDs
        seq_ids = jnp.array([ids[aa] for aa in seq])
        # 2. Create Mask (1 for data, 0 for pad)
        mask = jnp.ones(len(seq_ids))
        
        # 3. Pad both to bucket_size
        pad_len = bucket_size - len(seq_ids)
        # We pad features with 0 and mask with 0
        padded_ids = jnp.pad(seq_ids, (0, pad_len), constant_values=0) 
        padded_mask = jnp.pad(mask, (0, pad_len), constant_values=0)
        
        batch_features.append(param_matrix[padded_ids])
        batch_masks.append(padded_mask)
        batch_ids.append(padded_ids)
        
    return jnp.stack(batch_features), jnp.stack(batch_masks), jnp.stack(batch_ids)

@partial(jax.jit, static_argnames=['kernel_size'])
def process_batch_features(ids, features, mask, kernel_size):
    # Calculate mask
    conv_mask = jax.lax.reduce_window(mask, 0.0, jax.lax.max, 
                                 (1, kernel_size), (1, 1), 'VALID')
    
    # Calculate features
    b_scd = batch_conv(features[:, :, CHARGE], scd, kernel_size)
    b_nc  = batch_conv(features[:, :, CHARGE], net_charge, kernel_size)
    b_hps = batch_conv(features[:, :, HPS1], mean_hydro, kernel_size)
    b_fcr = batch_conv(features[:, :, CHARGE], fcr, kernel_size)
    b_shd = batch_conv(features[:, :, HPS1], shd, kernel_size)
    b_asym = batch_conv(features[:, :, CHARGE], charge_asymmetry, kernel_size)
    b_sbcs = batch_conv(features[:, :, HPS1], sbcs, kernel_size)
    b_s_q = batch_conv(ids, s_q, kernel_size)
    b_s_all = batch_conv(ids, s_all, kernel_size)
    
    # Stack and mask
    stacked = jnp.stack([b_scd, b_nc, b_hps, b_fcr, b_shd, b_asym, b_sbcs, b_s_q, b_s_all], axis=-1)
    return stacked * conv_mask[:, :, None], conv_mask

seqs.sort(key=len)
batch_size = 5
kernel_size = 10
for i in range(0, len(seqs), batch_size):
    batch_list = seqs[i : i + batch_size]
    
    # 3. Find the largest protein in THIS specific batch
    # This minimizes "wasted" zeros!
    current_bucket_size = len(batch_list[-1]) 
    
    # 4. Process this batch
    # (Using the create_bucket_batch function we discussed)
    batch_features, batch_mask, batch_ids = create_bucket_batch(batch_list, current_bucket_size, param_matrix, ids)
    print(f"Batch {i//batch_size + 1}, contains {len(batch_list)} sequences:")
    print("Batch IDs shape:", batch_ids.shape)  # (batch_size, bucket_size)
    print("Batch Mask shape:", batch_mask.shape)  # (batch_size, bucket_size)
    print("Batch raw data shape:", batch_features.shape)  # (batch_size, bucket_size, num_features)

    # Execute
    print("Processing batch features... into convolution tensor")
    print("kernel_size:", kernel_size)
    features, conv_mask = process_batch_features(batch_ids, batch_features, batch_mask, kernel_size)
    print("physics calculated features shape:", features.shape)  # (batch_size, bucket_size, num_features)
    print("Mask shape:", conv_mask.shape)          # (batch_size, bucket_size)
    print("-" * 50)    

Batch 1, contains 5 sequences:
Batch IDs shape: (5, 100)
Batch Mask shape: (5, 100)
Batch raw data shape: (5, 100, 5)
Processing batch features... into convolution tensor
kernel_size: 10
physics calculated features shape: (5, 91, 9)
Mask shape: (5, 91)
--------------------------------------------------
Batch 2, contains 5 sequences:
Batch IDs shape: (5, 131)
Batch Mask shape: (5, 131)
Batch raw data shape: (5, 131, 5)
Processing batch features... into convolution tensor
kernel_size: 10
physics calculated features shape: (5, 122, 9)
Mask shape: (5, 122)
--------------------------------------------------
Batch 3, contains 2 sequences:
Batch IDs shape: (2, 140)
Batch Mask shape: (2, 140)
Batch raw data shape: (2, 140, 5)
Processing batch features... into convolution tensor
kernel_size: 10
physics calculated features shape: (2, 131, 9)
Mask shape: (2, 131)
--------------------------------------------------


In [ ]:



import jax
import jax.numpy as jnp
from jax.scipy.linalg import toeplitz
from functools import partial
import numpy as np

MASS = 0
CHARGE = 1
SIGMA = 2
HPS1 = 3
HPS2 = 4

AA_PARAMS = {
    "-": dict(mass=0.0, charge=0.0, sigma=0.0, HPS1=0.0, HPS2=0.0),  # Padding
    "A": dict(mass=71.08, charge=0.0, sigma=0.504, HPS1=0.730, HPS2=0.003),
    "R": dict(mass=156.20, charge=1.0, sigma=0.656, HPS1=0.000, HPS2=0.723),
    "N": dict(mass=114.10, charge=0.0, sigma=0.568, HPS1=0.432, HPS2=0.160),
    "D": dict(mass=115.10, charge=-1.0, sigma=0.558, HPS1=0.378, HPS2=0.002),
    "C": dict(mass=103.10, charge=0.0, sigma=0.548, HPS1=0.595, HPS2=0.400),
    "Q": dict(mass=128.10, charge=0.0, sigma=0.602, HPS1=0.514, HPS2=0.468),
    "E": dict(mass=129.10, charge=-1.0, sigma=0.592, HPS1=0.459, HPS2=0.022),
    "G": dict(mass=57.05, charge=0.0, sigma=0.450, HPS1=0.649, HPS2=0.784),
    "H": dict(mass=137.10, charge=0.5, sigma=0.608, HPS1=0.514, HPS2=0.487),
    "I": dict(mass=113.20, charge=0.0, sigma=0.618, HPS1=0.973, HPS2=0.687),
    "L": dict(mass=113.20, charge=0.0, sigma=0.618, HPS1=0.973, HPS2=0.335),
    "K": dict(mass=128.20, charge=1.0, sigma=0.636, HPS1=0.514, HPS2=0.095),
    "M": dict(mass=131.20, charge=0.0, sigma=0.618, HPS1=0.838, HPS2=0.993),
    "F": dict(mass=147.20, charge=0.0, sigma=0.636, HPS1=1.000, HPS2=0.871),
    "P": dict(mass=97.12, charge=0.0, sigma=0.556, HPS1=1.000, HPS2=0.471),
    "S": dict(mass=87.08, charge=0.0, sigma=0.518, HPS1=0.595, HPS2=0.487),
    "T": dict(mass=101.10, charge=0.0, sigma=0.562, HPS1=0.676, HPS2=0.274),
    "W": dict(mass=186.20, charge=0.0, sigma=0.678, HPS1=0.946, HPS2=0.753),
    "Y": dict(mass=163.20, charge=0.0, sigma=0.646, HPS1=0.865, HPS2=0.984),
    "V": dict(mass=99.07, charge=0.0, sigma=0.586, HPS1=0.892, HPS2=0.428),
}

# ── Derived arrays ───────────────────────────────────
# Protein + number id dictionary
ids = {aa: i for i, aa in enumerate(list(AA_PARAMS.keys()))}

param_matrix = jnp.array([
    [v["mass"], v["charge"], v["sigma"], v["HPS1"], v["HPS2"]] 
    for v in AA_PARAMS.values()
])

is_charged_mask = jnp.array([v["charge"] != 0 for v in AA_PARAMS.values()])

def s_all(window_ids):
    counts = jnp.bincount(window_ids, length=21)
    real_counts = counts[1:] 
    
    N_all = jnp.sum(real_counts)
    
    p_all = real_counts / jnp.where(N_all == 0, 1.0, N_all)
    safe_p_all = jnp.where(p_all > 0, p_all, 1.0)
    
    S_all = -jnp.sum(p_all * jnp.log2(safe_p_all))

    return jnp.where(N_all == 0, 0.0, S_all)

def s_q(window_ids):
    counts = jnp.bincount(window_ids, length=21)
    charged_counts = counts * is_charged_mask
    
    N_q = jnp.sum(charged_counts)
    
    p_q = charged_counts / jnp.where(N_q == 0, 1.0, N_q)
    safe_p_q = jnp.where(p_q > 0, p_q, 1.0)
    
    S_q = -jnp.sum(p_q * jnp.log2(safe_p_q))

    return jnp.where(N_q == 0, 0.0, S_q)

#sequence charge decoration (SCD)
def scd(window): 
    k = len(window)
    qij = jnp.outer(window,window)
    dist = toeplitz(jnp.arange(k)) ** 0.5
    S = qij * dist
    s_masked = jnp.tril(S,k=-1)
    return jnp.sum(s_masked)/k
#sequence hydropathy decoration (ShD)
def shd(window): 
    k = len(window)
    lambda_ij = window[:,None] + window[None,:]
    dist = toeplitz(jnp.arange(k))
    safe_dist = jnp.where(dist == 0, 1.0, dist)
    S = lambda_ij * (1/safe_dist)
    s_masked = jnp.tril(S,k=-1)
    return jnp.sum(s_masked)/k

def net_charge(window):
    return jnp.sum(window)

#Fraction of Charged Residues (FCR)
def fcr(window):
    return jnp.sum(jnp.abs(window) > 0) / len(window)

def charge_asymmetry(window):
    n = len(window)
    # Calculate fractions of positive and negative charges
    pos = jnp.sum(window > 0) / n
    neg = jnp.sum(window < 0) / n
    total = pos + neg
    
    # jnp.where(condition, value_if_true, value_if_false)
    return jnp.where(total == 0, 0.0, (pos - neg)**2 / total)

#Mean Hydrophobicity (using HPS1)
def mean_hydro(window):
    return jnp.mean(window)

def sbcs(window_lambda):
    is_high = window_lambda > 0.5
    mean_lambda = jnp.mean(window_lambda)

    # Get positions of high-lambda residues in order
    N = window_lambda.shape[0]
    distances = jnp.arange(N)
    # Positions of high-lambda residues; non-stickers get sentinel N
    sticker_pos = jnp.where(is_high, distances, N)
    sticker_pos_sorted = jnp.sort(sticker_pos)  # real stickers first, then N's
    
    # Consecutive differences between sticker positions
    # d_{j, j+1} = sticker_pos[j+1] - sticker_pos[j]
    diffs = sticker_pos_sorted[1:] - sticker_pos_sorted[:-1]
    
    # A pair is valid if both positions are real stickers (< N)
    valid = (sticker_pos_sorted[:-1] < N) & (sticker_pos_sorted[1:] < N)
    
    # Safe inverse distance (avoid div by zero on sentinel pairs)
    safe_diffs = jnp.where(valid, diffs, 1.0)
    inv_dist = jnp.where(valid, 1.0 / safe_diffs, 0.0)
    
    return mean_lambda * jnp.sum(inv_dist)

def conv(x, f, kernel_size):
    # This creates the (Num_Windows, Window_Size) matrix
    indices = jnp.arange(len(x)- kernel_size + 1)[:, None] + jnp.arange(kernel_size)
    windows = x[indices]
    return jax.vmap(f)(windows)

batch_conv = jax.vmap(conv, in_axes=(0, None, None))

def create_batch(protein_list,max_size,param_matrix, ids):
    """
    Takes a list of strings, pads them to  max size, 
    and returns features + mask.
    """
    batch_features = []
    batch_masks = []
    batch_ids = []

    for seq in protein_list:
        # 1. Convert to IDs
        seq_ids = jnp.array([ids[aa] for aa in seq])
        # 2. Create Mask (1 for data, 0 for pad)
        mask = jnp.ones(len(seq_ids))
        
        # 3. Pad both to bucket_size
        pad_len = max_size - len(seq_ids)
        # We pad features with 0 and mask with 0
        padded_ids = jnp.pad(seq_ids, (0, pad_len), constant_values=0) 
        padded_mask = jnp.pad(mask, (0, pad_len), constant_values=0)
        
        batch_features.append(param_matrix[padded_ids])
        batch_masks.append(padded_mask)
        batch_ids.append(padded_ids)
        
    return jnp.stack(batch_features), jnp.stack(batch_masks), jnp.stack(batch_ids)

@partial(jax.jit, static_argnames=['kernel_size'])
def process_batch_features(ids, features, mask, kernel_size):
    # Calculate mask
    conv_mask = jax.lax.reduce_window(mask, 0.0, jax.lax.max, 
                                 (1, kernel_size), (1, 1), 'VALID')
    
    # Calculate features
    b_scd = batch_conv(features[:, :, CHARGE], scd, kernel_size)
    b_nc  = batch_conv(features[:, :, CHARGE], net_charge, kernel_size)
    b_hps = batch_conv(features[:, :, HPS1], mean_hydro, kernel_size)
    b_fcr = batch_conv(features[:, :, CHARGE], fcr, kernel_size)
    b_shd = batch_conv(features[:, :, HPS1], shd, kernel_size)
    b_asym = batch_conv(features[:, :, CHARGE], charge_asymmetry, kernel_size)
    b_sbcs = batch_conv(features[:, :, HPS1], sbcs, kernel_size)
    b_s_q = batch_conv(ids, s_q, kernel_size)
    b_s_all = batch_conv(ids, s_all, kernel_size)
    
    # Stack and mask
    stacked = jnp.stack([b_scd, b_nc, b_hps, b_fcr, b_shd, b_asym, b_sbcs, b_s_q, b_s_all], axis=-1)
    return stacked * conv_mask[:, :, None], conv_mask
def log_kernels(k_min, k_max, n_channels):
    return [int(k) for k in np.unique(
        np.round(np.logspace(np.log10(k_min), np.log10(k_max), n_channels))
    )]

# For IDPs with sequences up to 500 residues:
kernels = log_kernels(k_min=5, k_max=200, n_channels=8)
seqs.sort(key=len)
max_size = len(seqs[-1])
kernel_sizes = [5,20,30,60]
batch_features, batch_mask, batch_ids = create_batch(seqs, max_size, param_matrix, ids)
print(f"Batch contains {len(seqs)} sequences:")
print("Batch IDs shape:", batch_ids.shape)  # (batch_size, bucket_size)
print("Batch Mask shape:", batch_mask.shape)  # (batch_size, bucket_size)
print("Batch raw data shape:", batch_features.shape)  # (batch_size, bucket_size, num_features)

print("Processing batch features... into convolution tensor channels ")
for kernel_size in kernel_sizes:
    # Execute
    print("kernel_size:", kernel_size)
    physics_tensor, conv_mask = process_batch_features(batch_ids, batch_features, batch_mask, kernel_size)
    print("physics calculated features shape:", physics_tensor.shape)  # (batch_size, bucket_size, num_features)
    print("Mask shape:", conv_mask.shape)       

NameError: name 'seq_params' is not defined